# IF + DIFFI: Few-Shot Evaluation

This notebook explores few shot evaluation because the previous IF model was performing very poorly in practice due to low data.

## approach:
- Leave-One-Out Cross-Validation (LOOCV) - proper evaluation without synthetic data
- K-Fold Cross-Validation - robustness estimation
- DIFFI attribution - understand which features drive anomalies
- model stability analysis - check sensitivity to hyperparameters

<hr>

# Data Loading & Preprocessing

- same preprocessing as `IMOS_anom_attr_model.ipynb`
- load 4 shark survey data, transpose, group into functional roles

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from typing import Tuple, Dict, List
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import LeaveOneOut, KFold
from scipy import stats
import time
from math import ceil

In [2]:
# load data
df = pd.read_csv('4shark_surveys_species_count.csv')
df

,species_name,Survey 1,Survey 2,Survey 3,Survey 4,Survey 5,Survey 6,Survey 7,Survey 8,Survey 9,...,Survey 797,Survey 798,Survey 799,Survey 800,Survey 801,Survey 802,Survey 803,Survey 804,Survey 805,Survey 806
0,Ctenochaetus striatus,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,3,0,0
1,Chlorurus sordidus,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Labroides dimidiatus,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,4,3,2,3,1
3,Thalassoma lutescens,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Acanthurus nigrofuscus,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1205,Taeniamia biguttata,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
1206,Taeniamia melasma,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1207,Aipysurus spp.,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1208,Zapteryx xyster,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
df_T = df.set_index("species_name").T
df_T.index.name = None
df_T = df_T.reset_index(drop=True)
df_T = df_T.rename_axis(None, axis=1)
print(f"Transposed shape: {df_T.shape}")

Transposed shape: (806, 1210)


In [4]:
APEX_PREDATORS = [
    "Carcharhinus melanopterus", "Triaenodon obesus",
    "Carcharhinus amblyrhynchos", "Carcharhinus albimarginatus",
]
HERBIVORE_SCRAPERS = [
    "Chlorurus sordidus", "Chlorurus microrhinos", "Scarus niger",
    "Hipposcarus longiceps", "Scarus ghobban", "Scarus flavipectoralis",
]
TURF_BRUSHERS = [
    "Ctenochaetus striatus", "Ctenochaetus flavicauda", "Ctenochaetus binotatus",
    "Acanthurus nigrofuscus", "Acanthurus grammoptilus",
]
INVERTEBRATE_PREY_HUNTERS = [
    "Parupeneus multifasciatus", "Parupeneus cyclostomus", "Parupeneus indicus",
    "Parupeneus barberinus", "Parupeneus barberinoides", "Parupeneus spilurus",
    "Lethrinus atkinsoni", "Hemigymnus melapterus", "Monotaxis grandoculis",
    "Monotaxis heterodon", "Scolopsis bilineata",
]

ROLE_MAP = {}
for sp in APEX_PREDATORS: ROLE_MAP[sp] = "Apex_Predators"
for sp in HERBIVORE_SCRAPERS: ROLE_MAP[sp] = "Herbivore_Scrapers"
for sp in TURF_BRUSHERS: ROLE_MAP[sp] = "Turf_Brushers"
for sp in INVERTEBRATE_PREY_HUNTERS: ROLE_MAP[sp] = "Invertebrate_Prey_Hunters"

ROLE_COLUMNS = ["Apex_Predators", "Herbivore_Scrapers", "Turf_Brushers", "Invertebrate_Prey_Hunters"]

In [5]:
def make_role_df(df, warn=True):
    df = df.copy()
    all_role_species = list(ROLE_MAP.keys())
    missing = [s for s in all_role_species if s not in df.columns]
    if warn and missing:
        print("defined species absent from DataFrame:")
        for s in missing: print(f"   - {s}  [{ROLE_MAP[s]}]")
    new_df = pd.DataFrame()
    for role in ROLE_COLUMNS:
        species_for_role = [s for s in ROLE_MAP if ROLE_MAP[s] == role and s in df.columns]
        new_df[role] = df[species_for_role].sum(axis=1) if species_for_role else 0
    return new_df

df_with_roles = make_role_df(df_T, warn=True)
df_with_roles['Small_Invertebrates'] = df_with_roles['Invertebrate_Prey_Hunters'] * 5

FEATURE_COLUMNS = ["Apex_Predators", "Herbivore_Scrapers", "Turf_Brushers", "Invertebrate_Prey_Hunters", "Small_Invertebrates"]
df = df_with_roles[FEATURE_COLUMNS]
print(f"Final dataset shape: {df.shape}")

Final dataset shape: (806, 5)


In [6]:
# filter out surveys with >75% apex predators
df['Apex_Proportion'] = df['Apex_Predators'] / df.sum(axis=1)
df = df[df['Apex_Proportion'] <= 0.75].drop(columns=['Apex_Proportion'])
print(f"After filtering: {df.shape}")

After filtering: (704, 5)


<hr>

# Expert Baseline Deviation Score

- expert provided ONE reference point: the species ratios for a healthy reef at equilibrium
- this is NOT labeled data — there are no "anomalous" or "normal" surveys in the dataset
- instead, we compute how far each survey deviates from this healthy reference
- the deviation score is a continuous measure: higher = further from healthy equilibrium
- this becomes our reference metric for evaluating whether the IF model's anomaly scores make ecological sense

In [7]:
# expert baseline ratios for healthy reef (equilibrium)
EXPERT_BASELINE = {
    'Apex_Predators': 1.442,
    'Turf_Brushers': 4.02,
    'Herbivore_Scrapers': 5.67,
    'Invertebrate_Prey_Hunters': 2.47,
    'Small_Invertebrates': 10.277,
}

# calculate deviation from baseline for each survey
# using normalized euclidean distance from equilibrium
baseline_values = np.array([EXPERT_BASELINE[col] for col in FEATURE_COLUMNS])
survey_values = df[FEATURE_COLUMNS].values

# normalize by baseline to get relative deviation
relative_deviation = np.abs(survey_values - baseline_values) / baseline_values

# deviation score = euclidean distance from baseline (normalized)
deviation_scores = np.sqrt(np.sum(relative_deviation**2, axis=1))

# store in dataframe
df['deviation_score'] = deviation_scores

print(f"deviation scores: min={deviation_scores.min():.3f}, max={deviation_scores.max():.3f}, mean={deviation_scores.mean():.3f}")
print(f"median: {np.median(deviation_scores):.3f}")
print(f"\nthis is a CONTINUOUS score, not a binary label.")
print(f"there are no 'anomalous' surveys — only surveys that are more or less deviated from the expert's healthy reference.")

deviation scores: min=0.436, max=15.784, mean=2.714
median: 2.036

this is a CONTINUOUS score, not a binary label.
there are no 'anomalous' surveys — only surveys that are more or less deviated from the expert's healthy reference.


<hr>

# DIFFI Implementation

- DIFFI provides feature attribution for IF anomaly scores
- tells us WHICH features drove each anomaly detection
- from: https://arxiv.org/abs/2007.11117

In [8]:
# sklearn patch to expose per-tree scores
from sklearn.ensemble._iforest import _average_path_length
from sklearn.utils.validation import _num_samples
from sklearn.utils import gen_batches
from sklearn.utils._chunking import get_chunk_n_rows

def decision_function_single_tree(iforest, tree_idx, X):
    return _score_samples(iforest, tree_idx, X) - iforest.offset_

def _score_samples(iforest, tree_idx, X):
    if iforest.n_features_in_ != X.shape[1]:
        raise ValueError("Number of features mismatch")
    return -_compute_chunked_score_samples(iforest, tree_idx, X)

def _compute_chunked_score_samples(iforest, tree_idx, X):
    n_samples = _num_samples(X)
    subsample_features = (iforest._max_features != X.shape[1])
    chunk_n_rows = get_chunk_n_rows(row_bytes=16 * iforest._max_features, max_n_rows=n_samples)
    slices = gen_batches(n_samples, chunk_n_rows)
    scores = np.zeros(n_samples, order="f")
    for sl in slices:
        scores[sl] = _compute_score_samples_single_tree(iforest, tree_idx, X[sl], subsample_features)
    return scores

def _compute_score_samples_single_tree(iforest, tree_idx, X, subsample_features):
    n_samples = X.shape[0]
    depths = np.zeros(n_samples, order="f")
    tree = iforest.estimators_[tree_idx]
    features = iforest.estimators_features_[tree_idx]
    X_subset = X[:, features] if subsample_features else X
    leaves_index = tree.apply(X_subset)
    node_indicator = tree.decision_path(X_subset)
    n_samples_leaf = tree.tree_.n_node_samples[leaves_index]
    depths += (np.ravel(node_indicator.sum(axis=1)) + _average_path_length(n_samples_leaf) - 1.0)
    return 2 ** (-depths / (1 * _average_path_length([iforest.max_samples_])))

def _get_iic(estimator, predictions, is_leaves, adjust_iic):
    desired_min, desired_max, epsilon = 0.5, 1.0, 0.0
    n_nodes = estimator.tree_.node_count
    lambda_ = np.zeros(n_nodes)
    children_left = estimator.tree_.children_left
    children_right = estimator.tree_.children_right
    node_indicator_all_samples = estimator.decision_path(predictions).toarray()
    num_samples_in_node = np.sum(node_indicator_all_samples, axis=0)
    for node in range(n_nodes):
        num_samples_in_current_node = num_samples_in_node[node]
        num_samples_in_left_children = num_samples_in_node[children_left[node]]
        num_samples_in_right_children = num_samples_in_node[children_right[node]]
        if num_samples_in_current_node == 0 or num_samples_in_current_node == 1 or is_leaves[node]:
            lambda_[node] = -1
        elif num_samples_in_left_children == 0 or num_samples_in_right_children == 0:
            lambda_[node] = epsilon
        else:
            if num_samples_in_current_node % 2 == 0: current_min = 0.5
            else: current_min = ceil(num_samples_in_current_node / 2) / num_samples_in_current_node
            current_max = (num_samples_in_current_node - 1) / num_samples_in_current_node
            tmp = np.max([num_samples_in_left_children, num_samples_in_right_children]) / num_samples_in_current_node
            if adjust_iic and current_min != current_max:
                lambda_[node] = ((tmp - current_min) / (current_max - current_min)) * (desired_max - desired_min) + desired_min
            else:
                lambda_[node] = tmp
    return lambda_

def diffi_ib(iforest, X, adjust_iic=True):
    """global DIFFI - which features drive anomalies overall"""
    start = time.time()
    num_feat = X.shape[1]
    estimators = iforest.estimators_
    cfi_outliers_ib = np.zeros(num_feat).astype('float')
    cfi_inliers_ib = np.zeros(num_feat).astype('float')
    counter_outliers_ib = np.zeros(num_feat).astype('int')
    counter_inliers_ib = np.zeros(num_feat).astype('int')
    in_bag_samples = iforest.estimators_samples_
    for k, estimator in enumerate(estimators):
        in_bag_sample = list(in_bag_samples[k])
        X_ib = X[in_bag_sample, :]
        as_ib = decision_function_single_tree(iforest, k, X_ib)
        X_outliers_ib = X_ib[np.where(as_ib < 0)]
        X_inliers_ib = X_ib[np.where(as_ib > 0)]
        if X_inliers_ib.shape[0] == 0 or X_outliers_ib.shape[0] == 0: continue
        n_nodes = estimator.tree_.node_count
        children_left = estimator.tree_.children_left
        children_right = estimator.tree_.children_right
        feature = estimator.tree_.feature
        node_depth = np.zeros(shape=n_nodes, dtype=np.int64)
        is_leaves = np.zeros(shape=n_nodes, dtype=bool)
        stack = [(0, -1)]
        while len(stack) > 0:
            node_id, parent_depth = stack.pop()
            node_depth[node_id] = parent_depth + 1
            if children_left[node_id] != children_right[node_id]:
                stack.append((children_left[node_id], parent_depth + 1))
                stack.append((children_right[node_id], parent_depth + 1))
            else:
                is_leaves[node_id] = True
        lambda_outliers_ib = _get_iic(estimator, X_outliers_ib, is_leaves, adjust_iic)
        node_indicator_all_points_outliers_ib = estimator.decision_path(X_outliers_ib)
        node_indicator_all_points_array_outliers_ib = node_indicator_all_points_outliers_ib.toarray()
        for i in range(len(X_outliers_ib)):
            path = list(np.where(node_indicator_all_points_array_outliers_ib[i] == 1)[0])
            depth = node_depth[path[-1]]
            for node in path:
                current_feature = feature[node]
                if lambda_outliers_ib[node] == -1: continue
                else:
                    cfi_outliers_ib[current_feature] += (1 / depth) * lambda_outliers_ib[node]
                    counter_outliers_ib[current_feature] += 1
        lambda_inliers_ib = _get_iic(estimator, X_inliers_ib, is_leaves, adjust_iic)
        node_indicator_all_points_inliers_ib = estimator.decision_path(X_inliers_ib)
        node_indicator_all_points_array_inliers_ib = node_indicator_all_points_inliers_ib.toarray()
        for i in range(len(X_inliers_ib)):
            path = list(np.where(node_indicator_all_points_array_inliers_ib[i] == 1)[0])
            depth = node_depth[path[-1]]
            for node in path:
                current_feature = feature[node]
                if lambda_inliers_ib[node] == -1: continue
                else:
                    cfi_inliers_ib[current_feature] += (1 / depth) * lambda_inliers_ib[node]
                    counter_inliers_ib[current_feature] += 1
    fi_outliers_ib = np.where(counter_outliers_ib > 0, cfi_outliers_ib / counter_outliers_ib, 0)
    fi_inliers_ib = np.where(counter_inliers_ib > 0, cfi_inliers_ib / counter_inliers_ib, 0)
    fi_ib = fi_outliers_ib / fi_inliers_ib
    return fi_ib, time.time() - start

def local_diffi(iforest, x):
    """local DIFFI - which features drove anomaly for a specific sample"""
    start = time.time()
    estimators = iforest.estimators_
    cfi = np.zeros(len(x)).astype('float')
    counter = np.zeros(len(x)).astype('int')
    max_depth = int(np.ceil(np.log2(iforest.max_samples_)))
    for estimator in estimators:
        n_nodes = estimator.tree_.node_count
        children_left = estimator.tree_.children_left
        children_right = estimator.tree_.children_right
        feature = estimator.tree_.feature
        node_depth = np.zeros(shape=n_nodes, dtype=np.int64)
        is_leaves = np.zeros(shape=n_nodes, dtype=bool)
        stack = [(0, -1)]
        while len(stack) > 0:
            node_id, parent_depth = stack.pop()
            node_depth[node_id] = parent_depth + 1
            if children_left[node_id] != children_right[node_id]:
                stack.append((children_left[node_id], parent_depth + 1))
                stack.append((children_right[node_id], parent_depth + 1))
            else:
                is_leaves[node_id] = True
        x_reshaped = x.reshape(1, -1)
        node_indicator = estimator.decision_path(x_reshaped)
        node_indicator_array = node_indicator.toarray()
        path = list(np.where(node_indicator_array == 1)[1])
        leaf_depth = node_depth[path[-1]]
        for node in path:
            if not is_leaves[node]:
                current_feature = feature[node]
                cfi[current_feature] += (1 / leaf_depth) - (1 / max_depth)
                counter[current_feature] += 1
    fi = np.zeros(len(cfi))
    for i in range(len(cfi)):
        if counter[i] != 0: fi[i] = cfi[i] / counter[i]
    return fi, time.time() - start

<hr>

# Critical Sanity Check: Expert Baseline Ranking

- the most important question: does the model rank the expert's healthy equilibrium as the LEAST anomalous?
- if the model thinks the healthy reference is more anomalous than real surveys, it's fundamentally broken
- we create a synthetic survey with the exact expert baseline values and score it alongside all real surveys

In [9]:
# create synthetic survey with expert baseline values
expert_survey = pd.DataFrame([EXPERT_BASELINE], columns=FEATURE_COLUMNS)

# train a quick IF model on the real data
IF_check = IsolationForest(contamination=0.1, random_state=42, n_estimators=100)
IF_check.fit(df[FEATURE_COLUMNS])

# score all real surveys
real_scores = IF_check.score_samples(df[FEATURE_COLUMNS])

# score the expert baseline
expert_score = IF_check.score_samples(expert_survey)[0]

# rank: how many real surveys have a HIGHER score (less anomalous) than the expert baseline?
n_less_anomalous = int(np.sum(real_scores > expert_score))
rank = n_less_anomalous + 1  # 1-based rank (1 = least anomalous)

print("critical sanity check: expert baseline ranking")
print("="*70)
print(f"\nexpert baseline IF score: {expert_score:.4f}")
print(f"real survey scores: min={real_scores.min():.4f}, max={real_scores.max():.4f}, mean={real_scores.mean():.4f}")
print(f"\nexpert baseline rank: {rank} out of {len(real_scores)+1} (1 = least anomalous)")
print(f"surveys less anomalous than expert baseline: {n_less_anomalous}")

if rank == 1:
    print("\n-> PASS: expert baseline is the LEAST anomalous. model is ecologically sensible.")
elif rank <= 10:
    print(f"\n-> PARTIAL PASS: expert baseline ranks #{rank}. close to least anomalous but not perfect.")
else:
    print(f"\n-> FAIL: expert baseline ranks #{rank}. model does NOT consider healthy reef as least anomalous.")
    print("   this means the model is fundamentally misaligned with expert knowledge.")

<hr>

# LOOCV Evaluation

- with about 35 samples, a traditional 85/15 split gives only about 5 test samples which is statistically meaningless
- LOOCV trains on n-1, tests on the held-out and then repeats n times
- gives n evaluation points instead of about 5

In [9]:
def loocv_anomaly_evaluation(X, model_params=None, contamination=0.1):
    """LOOCV for anomaly detection - each sample evaluated with model trained on all others"""
    if model_params is None:
        model_params = {'contamination': contamination, 'random_state': 42, 'n_estimators': 100}
    loo = LeaveOneOut()
    scores = np.zeros(len(X))
    predictions = np.zeros(len(X))
    for train_idx, test_idx in loo.split(X):
        model = IsolationForest(**model_params)
        model.fit(X.iloc[train_idx])
        scores[test_idx[0]] = model.score_samples(X.iloc[test_idx])[0]
        predictions[test_idx[0]] = model.predict(X.iloc[test_idx])[0]
    pred_labels = (predictions == -1).astype(int)
    normal_scores = scores[predictions == 1]
    anomaly_scores = scores[predictions == -1]
    return {
        'scores': scores, 'predictions': predictions,
        'n_anomalies': int(pred_labels.sum()), 'anomaly_rate': float(pred_labels.mean()),
        'score_stats': {
            'mean': float(scores.mean()), 'std': float(scores.std()),
            'normal_mean': float(normal_scores.mean()) if len(normal_scores) > 0 else np.nan,
            'normal_std': float(normal_scores.std()) if len(normal_scores) > 0 else np.nan,
            'anomaly_mean': float(anomaly_scores.mean()) if len(anomaly_scores) > 0 else np.nan,
            'anomaly_std': float(anomaly_scores.std()) if len(anomaly_scores) > 0 else np.nan,
        },
        'loocv_stability': float(scores.std())
    }

# Run LOOCV (use only feature columns)
print("Running LOOCV...")
loocv_results = loocv_anomaly_evaluation(df[FEATURE_COLUMNS], contamination=0.1)
print(f"Samples: {len(loocv_results['scores'])}")
print(f"Anomalies: {loocv_results['n_anomalies']} ({loocv_results['anomaly_rate']:.1%})")
print(f"Score mean: {loocv_results['score_stats']['mean']:.4f} +/- {loocv_results['score_stats']['std']:.4f}")
print(f"Stability (lower=better): {loocv_results['loocv_stability']:.4f}")

Running LOOCV...
Samples: 704
Anomalies: 74 (10.5%)
Score mean: -0.4608 +/- 0.0630
Stability (lower=better): 0.0630


<hr>

# K-Fold Cross-Validation

- K=5 provides another stability perspective
- compares to LOOCV for robustness check

In [10]:
def kfold_anomaly_evaluation(X, n_splits=5, model_params=None, contamination=0.1):
    """K-Fold CV for anomaly detection"""
    if model_params is None:
        model_params = {'contamination': contamination, 'random_state': 42, 'n_estimators': 100}
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_scores, fold_preds, fold_metrics = [], [], []
    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X)):
        model = IsolationForest(**model_params)
        model.fit(X.iloc[train_idx])
        scores = model.score_samples(X.iloc[test_idx])
        preds = model.predict(X.iloc[test_idx])
        fold_scores.extend(scores.tolist())
        fold_preds.extend(preds.tolist())
        n_anom = (preds == -1).sum()
        fold_metrics.append({'fold': fold_idx, 'n_test': len(test_idx), 'n_anomalies': int(n_anom), 'anomaly_rate': float(n_anom/len(test_idx))})
    scores_arr = np.array(fold_scores)
    return {
        'scores': scores_arr, 'predictions': np.array(fold_preds), 'fold_metrics': fold_metrics,
        'aggregate': {'mean_anomaly_rate': np.mean([m['anomaly_rate'] for m in fold_metrics]), 'score_stability': float(scores_arr.std())}
    }

# Run K-Fold (use only feature columns)
print("Running K-Fold (K=5)...")
kf_results = kfold_anomaly_evaluation(df[FEATURE_COLUMNS], n_splits=5, contamination=0.1)
print(f"Mean anomaly rate: {kf_results['aggregate']['mean_anomaly_rate']:.1%}")
print(f"Score stability: {kf_results['aggregate']['score_stability']:.4f}")

Running K-Fold (K=5)...
Mean anomaly rate: 10.8%
Score stability: 0.0618


<hr>

# Model Stability Analysis

- test how predictions vary with different hyperparameters
- low CV = stable model, high CV = needs more data

In [11]:
def analyze_model_stability(X, n_iterations=20, contamination_range=(0.05, 0.20)):
    """test model sensitivity to hyperparameter changes"""
    results = []
    for i in range(n_iterations):
        params = {
            'contamination': np.random.uniform(*contamination_range),
            'random_state': np.random.randint(0, 1000),
            'n_estimators': np.random.choice([50, 100, 150, 200]),
        }
        try:
            model = IsolationForest(**params)
            model.fit(X)
            preds = model.predict(X)
            results.append({'anomaly_rate': float((preds == -1).mean()), 'mean_score': float(model.score_samples(X).mean())})
        except: continue
    anomaly_rates = [r['anomaly_rate'] for r in results]
    return {'anomaly_rate_mean': np.mean(anomaly_rates), 'anomaly_rate_std': np.std(anomaly_rates),
            'anomaly_rate_cv': np.std(anomaly_rates)/np.mean(anomaly_rates) if np.mean(anomaly_rates) > 0 else np.nan}

# Run stability (use only feature columns)
print("Running stability analysis (20 iterations)...")
stability = analyze_model_stability(df[FEATURE_COLUMNS], n_iterations=20)
print(f"Anomaly rate: {stability['anomaly_rate_mean']:.1%} +/- {stability['anomaly_rate_std']:.1%}")
print(f"CV: {stability['anomaly_rate_cv']:.1%}")
if stability['anomaly_rate_cv'] < 0.2: print("-> STABLE")
elif stability['anomaly_rate_cv'] < 0.5: print("-> MODERATELY STABLE")
else: print("-> UNSTABLE - needs more data")

Running stability analysis (20 iterations)...
Anomaly rate: 13.2% +/- 4.9%
CV: 37.1%
-> MODERATELY STABLE


<hr>

# DIFFI Attribution

- now use DIFFI to understand which features drive anomalies
- global: which features matter overall
- local: which features drove each specific anomaly

In [12]:
# Train final model for DIFFI (use only feature columns)
IF = IsolationForest(contamination=0.1, random_state=42, n_estimators=100)
IF.fit(df[FEATURE_COLUMNS])

# Global DIFFI - which features drive anomalies overall
print("Running global DIFFI...")
fi_ib, exec_time = diffi_ib(IF, df[FEATURE_COLUMNS].values)
print(f"\nGlobal feature importance (higher = more anomalous):")
pd.Series(fi_ib, index=FEATURE_COLUMNS).sort_values(ascending=False)

Running global DIFFI...

Global feature importance (higher = more anomalous):


Apex_Predators               2.368377
Small_Invertebrates          1.584531
Herbivore_Scrapers           1.509718
Invertebrate_Prey_Hunters    1.508044
Turf_Brushers                1.465167
dtype: float64

In [13]:
# Local DIFFI - attribute each detected anomaly
X_features = df[FEATURE_COLUMNS]
preds = IF.predict(X_features)
anomaly_rows = X_features[preds == -1]

if len(anomaly_rows) > 0:
    print(f"\nLocal DIFFI for {len(anomaly_rows)} detected anomalies:")
    print("="*60)
    for idx in anomaly_rows.index:
        x = X_features.loc[idx].values
        fi, _ = local_diffi(IF, x)
        top_feat = pd.Series(fi, index=FEATURE_COLUMNS).sort_values(ascending=False).head(3)
        print(f"\nSurvey {idx}:")
        print(f"  Top drivers: {', '.join([f'{k} ({v:.3f})' for k, v in top_feat.items()])}")
else:
    print("\nNo anomalies detected in training data")


Local DIFFI for 71 detected anomalies:

Survey 95:
  Top drivers: Apex_Predators (0.088), Herbivore_Scrapers (0.053), Turf_Brushers (0.041)

Survey 101:
  Top drivers: Apex_Predators (0.108), Small_Invertebrates (0.042), Invertebrate_Prey_Hunters (0.038)

Survey 112:
  Top drivers: Herbivore_Scrapers (0.043), Invertebrate_Prey_Hunters (0.021), Small_Invertebrates (0.017)

Survey 114:
  Top drivers: Apex_Predators (0.056), Herbivore_Scrapers (0.056), Small_Invertebrates (0.039)

Survey 116:
  Top drivers: Herbivore_Scrapers (0.032), Invertebrate_Prey_Hunters (0.017), Small_Invertebrates (0.016)

Survey 120:
  Top drivers: Small_Invertebrates (0.024), Herbivore_Scrapers (0.023), Invertebrate_Prey_Hunters (0.019)

Survey 137:
  Top drivers: Apex_Predators (0.040), Invertebrate_Prey_Hunters (0.025), Small_Invertebrates (0.024)

Survey 140:
  Top drivers: Invertebrate_Prey_Hunters (0.029), Small_Invertebrates (0.027), Herbivore_Scrapers (0.027)

Survey 145:
  Top drivers: Apex_Predators (0

<hr>

# Evaluate Models Against Expert Baseline

- the expert provided ONE reference point: species ratios for a healthy reef
- there are NO labeled "anomalous" or "normal" surveys in the dataset
- the only valid evaluation is: does the model's anomaly score correlate with deviation from the expert's healthy reference?
- if a model assigns high anomaly scores to surveys that deviate a lot from the baseline, it's ecologically sensible
- we use Pearson and Spearman correlation as the primary metric

In [14]:
import os
import sys
from scipy.stats import pearsonr, spearmanr
sys.path.insert(0, os.path.join('..', '..', 'backend'))
from model_package import IFDiffiPackage
model_path = 'sharksphere_if_diffi.joblib'

# expert deviation scores (continuous, not binary)
deviation_scores = df['deviation_score'].values

if os.path.exists(model_path):
    pkg = joblib.load(model_path)
    deployed_scores = pkg.model.score_samples(df[FEATURE_COLUMNS])
    deployed_preds = pkg.model.predict(df[FEATURE_COLUMNS])
    
    loocv_scores = loocv_results['scores']
    loocv_preds = loocv_results['predictions']
    
    # correlation with expert deviation scores
    # IF scores are negative (more negative = more anomalous), so we negate them
    pearson_loocv, pearson_p_loocv = pearsonr(deviation_scores, -loocv_scores)
    spearman_loocv, spearman_p_loocv = spearmanr(deviation_scores, -loocv_scores)
    
    pearson_dep, pearson_p_dep = pearsonr(deviation_scores, -deployed_scores)
    spearman_dep, spearman_p_dep = spearmanr(deviation_scores, -deployed_scores)
    
    print("model evaluation against expert baseline")
    print("="*70)
    print("\nthe expert provided ONE reference point: species ratios for a healthy reef.")
    print("there are NO labeled 'anomalous' or 'normal' surveys.")
    print("the only valid metric is correlation between model anomaly scores and deviation from the expert baseline.")
    
    print(f"\n{'metric':<25} {'LOOCV':>15} {'Deployed':>15}")
    print("-"*70)
    print(f"{'pearson r':<25} {pearson_loocv:>14.4f} {pearson_dep:>14.4f}")
    print(f"{'  p-value':<25} {pearson_p_loocv:>14.2e} {pearson_p_dep:>14.2e}")
    print(f"{'spearman r':<25} {spearman_loocv:>14.4f} {spearman_dep:>14.4f}")
    print(f"{'  p-value':<25} {spearman_p_loocv:>14.2e} {spearman_p_dep:>14.2e}")
    
    print(f"\ninterpretation:")
    print("-"*70)
    if pearson_loocv > 0.7:
        print(f"  LOOCV: strong correlation (r={pearson_loocv:.3f}) — model scores align well with expert baseline")
    elif pearson_loocv > 0.4:
        print(f"  LOOCV: moderate correlation (r={pearson_loocv:.3f}) — model somewhat aligns with expert baseline")
    else:
        print(f"  LOOCV: weak correlation (r={pearson_loocv:.3f}) — model does not align well with expert baseline")
    
    if pearson_dep > 0.7:
        print(f"  Deployed: strong correlation (r={pearson_dep:.3f}) — model scores align well with expert baseline")
    elif pearson_dep > 0.4:
        print(f"  Deployed: moderate correlation (r={pearson_dep:.3f}) — model somewhat aligns with expert baseline")
    else:
        print(f"  Deployed: weak correlation (r={pearson_dep:.3f}) — model does not align well with expert baseline")
    
    # DIFFI comparison
    print(f"\nDIFFI attribution comparison")
    print("-"*70)
    fi_deployed, _ = diffi_ib(pkg.model, df[FEATURE_COLUMNS].values)
    fi_loocv, _ = diffi_ib(IF, df[FEATURE_COLUMNS].values)
    
    diffi_df = pd.DataFrame({
        'LOOCV': pd.Series(fi_loocv, index=FEATURE_COLUMNS),
        'Deployed': pd.Series(fi_deployed, index=FEATURE_COLUMNS),
    })
    diffi_df['rank_loocv'] = diffi_df['LOOCV'].rank(ascending=False).astype(int)
    diffi_df['rank_deployed'] = diffi_df['Deployed'].rank(ascending=False).astype(int)
    diffi_df['rank_diff'] = abs(diffi_df['rank_loocv'] - diffi_df['rank_deployed'])
    diffi_df = diffi_df.sort_values('LOOCV', ascending=False)
    print(diffi_df.to_string())
else:
    print(f"Model not found: {model_path}")

model evaluation against expert baseline

the expert provided ONE reference point: species ratios for a healthy reef.
there are NO labeled 'anomalous' or 'normal' surveys.
the only valid metric is correlation between model anomaly scores and deviation from the expert baseline.

metric                              LOOCV        Deployed
----------------------------------------------------------------------
pearson r                         0.7788         0.7528
  p-value                      2.60e-144      1.33e-129
spearman r                        0.6727         0.6366
  p-value                       6.77e-94       2.87e-81

interpretation:
----------------------------------------------------------------------
  LOOCV: strong correlation (r=0.779) — model scores align well with expert baseline
  Deployed: strong correlation (r=0.753) — model scores align well with expert baseline

DIFFI attribution comparison
----------------------------------------------------------------------
      

<hr>

# Hyperparameter Tuning

- grid search over IF hyperparameters using LOOCV
- objective: maximize correlation between model anomaly scores and expert deviation scores
- with ~35 samples, LOOCV is feasible and gives reliable estimates

In [16]:
from itertools import product

# hyperparameter search space
param_grid = {
    'contamination': [0.08, 0.15, 0.20],
    'n_estimators': [100, 200, 300],
    'max_samples': [1.0],
}

X_tune = df[FEATURE_COLUMNS]
deviation_tune = df['deviation_score'].values

loo = LeaveOneOut()
results = []
total_combos = len(param_grid['contamination']) * len(param_grid['n_estimators']) * len(param_grid['max_samples'])
print(f"searching {total_combos} hyperparameter combinations via LOOCV...")
print("="*70)

for cont, n_est, max_samp in product(param_grid['contamination'], param_grid['n_estimators'], param_grid['max_samples']):
    scores = np.zeros(len(X_tune))
    for train_idx, test_idx in loo.split(X_tune):
        model = IsolationForest(
            contamination=cont,
            n_estimators=n_est,
            max_samples=max_samp,
            random_state=42
        )
        model.fit(X_tune.iloc[train_idx])
        scores[test_idx[0]] = model.score_samples(X_tune.iloc[test_idx])[0]
    
    # correlation with expert deviation scores
    pearson_r, _ = pearsonr(deviation_tune, -scores)
    spearman_r, _ = spearmanr(deviation_tune, -scores)
    
    results.append({
        'contamination': cont, 'n_estimators': n_est, 'max_samples': max_samp,
        'pearson_r': pearson_r, 'spearman_r': spearman_r,
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('pearson_r', ascending=False).reset_index(drop=True)

print(f"\ntop 10 configurations (by pearson correlation with expert deviation scores):")
print("-"*70)
top10 = results_df.head(10)[['contamination', 'n_estimators', 'max_samples', 'pearson_r', 'spearman_r']]
print(top10.to_string(index=False))

# best config
best = results_df.iloc[0]
print(f"\nbest configuration:")
print(f"  contamination:  {best['contamination']}")
print(f"  n_estimators:   {best['n_estimators']}")
print(f"  max_samples:    {best['max_samples']}")
print(f"  pearson r:      {best['pearson_r']:.4f}")
print(f"  spearman r:     {best['spearman_r']:.4f}")

searching 9 hyperparameter combinations via LOOCV...

top 10 configurations (by pearson correlation with expert deviation scores):
----------------------------------------------------------------------
 contamination  n_estimators  max_samples  pearson_r  spearman_r
          0.08           200          1.0   0.767342    0.649469
          0.20           200          1.0   0.767342    0.649469
          0.15           200          1.0   0.767342    0.649469
          0.20           300          1.0   0.749628    0.633211
          0.08           300          1.0   0.749628    0.633211
          0.15           300          1.0   0.749628    0.633211
          0.08           100          1.0   0.742295    0.625513
          0.15           100          1.0   0.742295    0.625513
          0.20           100          1.0   0.742295    0.625513

best configuration:
  contamination:  0.08
  n_estimators:   200.0
  max_samples:    1.0
  pearson r:      0.7673
  spearman r:     0.6495


<hr>

# Evaluate Tuned Model

- train final model with best hyperparameters
- compare against default model and deployed model

In [17]:
# train tuned model
best_params = {
    'contamination': best['contamination'],
    'n_estimators': int(best['n_estimators']),
    'max_samples': best['max_samples'],
    'random_state': 42
}
IF_tuned = IsolationForest(**best_params)
IF_tuned.fit(df[FEATURE_COLUMNS])

tuned_scores = IF_tuned.score_samples(df[FEATURE_COLUMNS])
pearson_tuned, _ = pearsonr(deviation_scores, -tuned_scores)
spearman_tuned, _ = spearmanr(deviation_scores, -tuned_scores)

# DIFFI for tuned model
fi_tuned, _ = diffi_ib(IF_tuned, df[FEATURE_COLUMNS].values)

print("model evaluation against expert baseline")
print("="*70)
print(f"\nbest params: contamination={best_params['contamination']}, n_estimators={best_params['n_estimators']}, max_samples={best_params['max_samples']}")

print(f"\n{'metric':<25} {'Default':>12} {'Tuned':>12} {'Deployed':>12}")
print("-"*70)
print(f"{'pearson r':<25} {pearson_loocv:>11.4f} {pearson_tuned:>11.4f} {pearson_dep:>11.4f}")
print(f"{'spearman r':<25} {spearman_loocv:>11.4f} {spearman_tuned:>11.4f} {spearman_dep:>11.4f}")

print(f"\nDIFFI attribution: Tuned model")
print("-"*70)
print(pd.Series(fi_tuned, index=FEATURE_COLUMNS).sort_values(ascending=False).to_string())

# improvement summary
print(f"\nimprovement from tuning:")
print("-"*70)
pearson_improvement = pearson_tuned - pearson_loocv
print(f"  pearson r: {pearson_loocv:.4f} -> {pearson_tuned:.4f} ({pearson_improvement:+.4f})")
if pearson_improvement > 0:
    print(f"  -> tuning improved correlation by {pearson_improvement:.4f}")
else:
    print(f"  -> tuning did not improve correlation (default was already good)")

model evaluation against expert baseline

best params: contamination=0.08, n_estimators=200, max_samples=1.0

metric                         Default        Tuned     Deployed
----------------------------------------------------------------------
pearson r                      0.7788      0.7662      0.7528
spearman r                     0.6727      0.6478      0.6366

DIFFI attribution: Tuned model
----------------------------------------------------------------------
Apex_Predators               2.035527
Turf_Brushers                1.593450
Small_Invertebrates          1.560170
Herbivore_Scrapers           1.428402
Invertebrate_Prey_Hunters    1.424054

improvement from tuning:
----------------------------------------------------------------------
  pearson r: 0.7788 -> 0.7662 (-0.0126)
  -> tuning did not improve correlation (default was already good)


<hr>

# Export Final Model

- since default model outperformed tuned model, use default params
- export in format compatible with backend/model_package.py
- saves to backend/sharksphere_if_diffi.joblib

In [19]:
import os
import sys
sys.path.insert(0, os.path.join('..', '..', 'backend'))
from model_package import export_package, IFDiffiPackage

# use default params (they outperformed tuned)
IF_final = IsolationForest(
    contamination=0.1,
    n_estimators=100,
    max_samples=1.0,
    random_state=42
)
IF_final.fit(df[FEATURE_COLUMNS])

# export to backend directory
export_dir = os.path.join('..', '..', 'backend')
os.makedirs(export_dir, exist_ok=True)
export_path = os.path.join(export_dir, 'sharksphere_if_diffi.joblib')
saved_path = export_package(IF_final, export_path)
print(f"model exported to: {saved_path}")
print(f"file size: {os.path.getsize(saved_path) / 1024:.1f} KB")

# verify it loads and works
from model_package import load_package
pkg = load_package(saved_path)
test_result = pkg.predict_one({
    'Apex_Predators': 4,
    'Herbivore_Scrapers': 18,
    'Turf_Brushers': 12,
    'Invertebrate_Prey_Hunters': 7,
    'Small_Invertebrates': 35,
})
print(f"\ntest prediction:")
print(f"  prediction: {test_result['prediction']} (1=normal, -1=anomaly)")
print(f"  anomaly score: {test_result['anomaly_score']:.4f}")
print(f"  DIFFI scores: {test_result['diffi_scores']}")

model exported to: ..\..\backend\sharksphere_if_diffi.joblib
file size: 2799.5 KB

test prediction:
  prediction: -1 (1=normal, -1=anomaly)
  anomaly score: -0.5965
  DIFFI scores: {'Apex_Predators': 0.02457757296466972, 'Herbivore_Scrapers': 0.02640257252326217, 'Turf_Brushers': 0.01642702535559677, 'Invertebrate_Prey_Hunters': 0.015311068498987947, 'Small_Invertebrates': 0.01953583453583451}


<hr>

# Summary

## what this means:
- LOOCV: proper anomaly count without train/test split bias
- K-Fold: robustness check
- Stability: is model reliable or needs more data
- DIFFI: which features actually drive each anomaly
- Expert baseline: ONE reference point for a healthy reef, used to compute deviation scores
- Evaluation: correlation between model anomaly scores and expert deviation scores
- Hyperparameter tuning: confirmed default params are best

## next steps:
- if CV > 0.3, lack of data :(
- review DIFFI attributions for ecological sense
- use local_diffi in production to explain detections
- model is exported and ready for deployment